In [1]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm.auto import tqdm
import torch.nn as nn
import numpy as np 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import h5py
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
import src.spatial_attn_lightning as attn_tracking_lightning
import src.audio_transforms as at
import torch
import yaml

import argparse
from argparse import ArgumentParser
from corpus.speaker_room_dataset import SpeakerRoomDataset
from tqdm.auto import tqdm
from datetime import datetime
import sys 
import torchaudio.transforms as T

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
torch._dynamo.config.skip_nnmodule_hook_guards=False

In [3]:
config_path = "config/binaural_attn/word_task_half_co_loc_v07.yaml"
ckpt_path = "attn_cue_models/word_task_half_co_loc_v07/checkpoints/epoch=2-step=46074.ckpt"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
# config['model']['new_modulee']=Tr
# config['getting_acts']  = True
model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config)


num_classes={'num_words': 800}


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [4]:
model_name = pathlib.Path(config_path).stem


In [5]:
from corpus.jsinV3_attn_tracking_multi_talker_background import jsinV3_attn_tracking_multi_talker_background
import src.audio_transforms as at

coch_gram = model.coch_gram.cuda()
label_type = 'CV'
sr = config['audio']['rep_kwargs']['sr']
config['data'] = {}
config['data']['corpus'] = {}
config['data']['corpus']['n_talkers'] = 1
config['data']['corpus']['root'] = '/om/user/imgriff/datasets/dataset_word_speaker_noise/JSIN_all_v3/subsets/' # New path to JSIN dataset


snr=0
audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.CombineWithRandomDBSNR(low_snr=-snr, high_snr=snr), 
                at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                at.UnsqueezeAudio(dim=0),
                at.DuplicateChannel() # only need to copy channels here
        ])  


# set up label re-mapping from SWC to CV labels 
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/Auditory-Attention/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
# key is int, val is word
wsn_class_map = word_and_speaker_encodings['word_idx_to_word']
# key is word, val is int
cv_class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
# map wsn class int key to cv class int value 
class_remap = {ix:(cv_class_map[word] if word in cv_class_map else -1) for ix, word in wsn_class_map.items()}

# def upsample op
upsample = T.Resample(20_000, sr,
                    lowpass_filter_width=64,
                    rolloff=0.9475937167399596,
                    beta=14.769656459379492,
                    resampling_method='kaiser_window',
                    dtype=torch.float32)
                            
# get dataset
dataset = jsinV3_attn_tracking_multi_talker_background(**config['data']['corpus'],
                                            mode='val',
                                            transform=None,
                                            demo=True)

def collate_fn(batch):
    #apply transforsms to batch
    cues = []
    foregrounds = []
    backgrounds = []
    mixtures = []
    labels = []
    for (fg, bg, cue, label) in batch:
        cue = audio_transforms(upsample(torch.from_numpy(cue)).squeeze(), None)[0]
        cues.append(cue)
        fg = upsample(torch.from_numpy(fg)).squeeze()
        bg = upsample(torch.from_numpy(bg)).squeeze()
        foregrounds.append(audio_transforms(fg, None)[0])
        backgrounds.append(audio_transforms(bg, None)[0])
        mixtures.append(audio_transforms(fg, bg)[0])
        labels.append(class_remap[label])

    cues = torch.stack(cues)
    foregrounds = torch.stack(foregrounds)
    backgrounds = torch.stack(backgrounds)
    mixtures = torch.stack(mixtures)
    labels = torch.tensor(labels).type(torch.LongTensor)
    return cues, foregrounds, backgrounds, mixtures, labels

dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=1,
            collate_fn=collate_fn
        )

1


In [6]:
list(model.model._orig_mod.model_dict.named_children() )

[('norm_coch_rep',
  LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)),
 ('attn_block_in', SimpleAttentionalGain()),
 ('conv_block_0',
  Sequential(
    (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
    (1): Conv2dSame(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
    (2): ReLU()
    (3): HannPooling2d()
  )),
 ('attn0', SimpleAttentionalGain()),
 ('conv_block_1',
  Sequential(
    (0): LayerNorm((32, 20, 5000), eps=1e-05, elementwise_affine=True)
    (1): Conv2dSame(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
    (2): ReLU()
    (3): HannPooling2d()
  )),
 ('attn1', SimpleAttentionalGain()),
 ('conv_block_2',
  Sequential(
    (0): LayerNorm((64, 10, 1250), eps=1e-05, elementwise_affine=True)
    (1): Conv2dSame(64, 256, kernel_size=(5, 5), stride=(1, 1), bias=False)
    (2): ReLU()
    (3): HannPooling2d()
  )),
 ('attn2', SimpleAttentionalGain()),
 ('conv_block_3',
  Sequential(
    (0): LayerNorm((256, 10, 250), eps=1e-05, elem

In [7]:
conv_modules = {name:module for name, module in model.model._orig_mod.model_dict.named_children() if 'conv' in name or 'relu' in name}
# add relu fc 
relu_fc = model.model._orig_mod.relufc
modules = {**conv_modules, **{'relufc': relu_fc}}

In [8]:
### Create module and forward hook that applys attnetional gains to activations
## Here we create an nn module that performs the same operations as SimpleAttentionalGain but only on the cue activations. 
## The parameters for each new module are taken from the existing checkpoint for each module in  model.attn_modules 

class AttentionalGains(nn.Module):
    def __init__(self, slope, bias, threshold):
        super(AttentionalGains, self).__init__()
        self.slope = slope.item()
        self.bias = bias.item()
        self.threshold = threshold.item()
        
    def forward(self, cue):
        cue = cue.mean(axis=-1,keepdim=True)
        # apply threshold shift
        cue = cue - self.threshold
        # apply slope
        cue = cue * self.slope
        # apply sigmoid & bias
        gain = self.bias + (1-self.bias) * torch.sigmoid(cue)
        return gain 
    
## Init gain modules per layer. 
gain_modules = {name:module for name,module in model.model.model_dict.items() if 'attn' in name}
gain_functions = {}


for name, module in gain_modules.items():
    gain_functions[name] = AttentionalGains(module.slope, module.bias, module.threshold)


In [9]:
## Add hooks to store attended stream 

# init array to store attended_acts
attended_acts = {}

def get_attention(name):
    def hook(model, input, output):
        # print(name)
        if name in attended_acts:
            attended_acts[name] = torch.cat((attended_acts[name], output.detach()), dim=0)
        else:
            attended_acts[name] = output.detach()
    return hook


# register hooks 
for name, module in gain_modules.items():
        module.register_forward_hook(get_attention(name)) 



In [10]:
model.attn_modules

[SimpleAttentionalGain(),
 SimpleAttentionalGain(),
 SimpleAttentionalGain(),
 SimpleAttentionalGain(),
 SimpleAttentionalGain(),
 SimpleAttentionalGain(),
 SimpleAttentionalGain(),
 SimpleAttentionalGain()]

In [11]:
# init array to store activations
activations = {}

def get_activation(name):
    def hook(model, input, output):
        # print(name)
        if name in activations:
            activations[name] = torch.cat((activations[name], output.detach()), dim=0)
        else:
            activations[name] = output.detach()
    return hook

# dicts to store activations
# fg_reps = {}
# bg_reps = {}
# mixture_reps = {}

# register hooks 
for name, module in modules.items():
    if 'relufc' in name:
        module.register_forward_hook(get_activation(name)) 
    else:
        module[0].register_forward_hook(get_activation(f"{name}_ln")) # [0] is layer norm 
        module[2].register_forward_hook(get_activation(f"{name}_relu")) # [2] is relu 
        module[3].register_forward_hook(get_activation(f"{name}_hannpool")) # [3] is pool 
    # lists to save acts in per-layer
    # fg_reps[name] = []
    # bg_reps[name] = []
    # mixture_reps[name] = []

# send model to gpu 
model = model.eval().cuda()


In [12]:
import h5py
from pathlib import Path
from scipy import stats 

In [13]:
# make map of module keys to gain keys 
len(modules.keys()), len(gain_modules.keys())

(8, 8)

In [18]:
activations['conv_block_0_hannpool']

torch.Size([2, 32, 20, 5000])

In [14]:
n_activations = 1 
# get activations 

# write acts to h5 
# outname = Path(f'binaural_model_attn_stage_reps/{model_name}/{model_name}_model_activations_0dB_w_cues_v2.h5')
# outname.parent.mkdir(parents=True, exist_ok=True)
# outname = 'test_act_outs.h5'
# with h5py.File(outname, 'w') as f:

with torch.no_grad():
    for ix, batch in tqdm(enumerate(dataloader),  total = n_activations):
        fg_cue, foreground, background, mixture, fg_target = batch

        # send to device
        foreground, background, mixture = foreground.cuda(), background.cuda(), mixture.cuda()
        fg_cue =  fg_cue.cuda()

        # convert to cochleagrams
        fg_cue, mixture = coch_gram(fg_cue, mixture)
        foreground, background = coch_gram(foreground, background)

        # if ix == 0:
        #         f.create_dataset('cochleagram_cue',shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_mixture', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_fg', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        #         f.create_dataset('cochleagram_bg', shape=[n_activations, mixture.view(-1).shape[0]], dtype=np.float32)
        # f['cochleagram_cue'][ix] = fg_cue.view(-1).cpu().numpy()
        # f['cochleagram_mixture'][ix] = mixture.view(-1).cpu().numpy()
        # f['cochleagram_fg'][ix] = foreground.view(-1).cpu().numpy()
        # f['cochleagram_bg'][ix] = background.view(-1).cpu().numpy()


        # run mixture
        activations = {}

        pred = model(fg_cue, mixture, None)
        for layer in attended_acts.keys():
             attended = attended_acts[layer]
        for layer in activations.keys():
            print(layer)
            if 'relufc' in layer:
                    mixture_acts = activations[layer] 
                    # if ix == 0:
                    # f.create_dataset(f'{layer}_mixture', shape=[n_activations, np.prod(mixture_acts.shape)], dtype=np.float32)
            else:
                cue_acts, mixture_acts = activations[layer] 
            # break 
            #     if ix == 0:
            #         f.create_dataset(f'{layer}_cue', shape=[n_activations, np.prod(cue_acts.shape)], dtype=np.float32)
            #         f.create_dataset(f'{layer}_mixture', shape=[n_activations, np.prod(mixture_acts.shape)], dtype=np.float32)
            #     f[f'{layer}_cue'][ix] = cue_acts.view(-1).cpu().numpy()
            # f[f'{layer}_mixture'][ix] = mixture_acts.view(-1).cpu().numpy()
        break
        # activations = {}
        # print(activatiions)
        
        # run fg - can skip cue 
        # pred = model(fg_cue, foreground, None)
        # for layer in activations.keys():
        #     if 'relufc' in layer:
        #             fg_acts = activations[layer] 
        #     else:
        #         _, fg_acts = activations[layer]
        #     # if ix == 0:
        #         # f.create_dataset(f'{layer}_fg', shape=[n_activations, np.prod(fg_acts.shape)], dtype=np.float32)
        #     # f[f'{layer}_fg'][ix] = fg_acts.view(-1).cpu().numpy()
        # activations = {}

        # # run bg
        # pred = model(fg_cue, background, None)
        # for layer in activations.keys():
        #     if 'relufc' in layer:
        #             bg_acts = activations[layer]
        #     else:
        #         _, bg_acts = activations[layer]
        #     # if ix == 0:
        #         # get flattened shape of bg_acts 
        #         # f.create_dataset(f'{layer}_bg', shape=[n_activations, np.prod(bg_acts.shape)], dtype=np.float32)
        #     # f[f'{layer}_bg'][ix] = bg_acts.view(-1).cpu().numpy()
        # activations = {}
        

        # Save signals without cue in forward pass 
        # run mixture
        # pred = model(None, mixture, None)
        # for layer in activations.keys():
        #     print(layer)
        #     mixture_acts = activations[layer] 
        #     break
        #     if ix == 0:
        #         f.create_dataset(f'{layer}_mixture_no_cue', shape=[n_activations, np.prod(mixture_acts.shape)], dtype=np.float32)
        #     f[f'{layer}_mixture_no_cue'][ix] = mixture_acts.view(-1).cpu().numpy()
        # # activations = {}
        # break
        # # run fg - can skip cue
        # pred = model(None, foreground, None)
        # for layer in activations.keys():
        #     fg_acts = activations[layer] 
        #     break
            # if ix == 0:
            #     f.create_dataset(f'{layer}_fg_no_cue', shape=[n_activations, np.prod(fg_acts.shape)], dtype=np.float32)
            # f[f'{layer}_fg_no_cue'][ix] = fg_acts.view(-1).cpu().numpy()
        # activations = {}

        # # run bg
        # pred = model(None, background, None)
        # for layer in activations.keys():
        #     bg_acts = activations[layer]
        #     # if ix == 0:
        #     #     f.create_dataset(f'{layer}_bg_no_cue', shape=[n_activations, np.prod(bg_acts.shape)], dtype=np.float32)
        #     # f[f'{layer}_bg_no_cue'][ix] = bg_acts.view(-1).cpu().numpy()
        # activations = {}

        # if ix == n_activations-1:
        #     break
      

  0%|          | 0/1 [00:13<?, ?it/s]

conv_block_0_ln
conv_block_0_relu
conv_block_0_hannpool
conv_block_1_ln
conv_block_1_relu
conv_block_1_hannpool
conv_block_2_ln
conv_block_2_relu
conv_block_2_hannpool
conv_block_3_ln
conv_block_3_relu
conv_block_3_hannpool
conv_block_4_ln
conv_block_4_relu
conv_block_4_hannpool
conv_block_5_ln
conv_block_5_relu
conv_block_5_hannpool
conv_block_6_ln
conv_block_6_relu
conv_block_6_hannpool
relufc


In [19]:
attended_acts['attn0'].shape

torch.Size([1, 32, 20, 5000])

{'attn_block_in': AttentionalGains(),
 'attn0': AttentionalGains(),
 'attn1': AttentionalGains(),
 'attn2': AttentionalGains(),
 'attn3': AttentionalGains(),
 'attn4': AttentionalGains(),
 'attn5': AttentionalGains(),
 'attn6': AttentionalGains()}

In [38]:
### see if manual application of gains gives same results as attended 

attended = attended_acts['attn0']

# get gains from cue 
cue_acts, mixture = activations['conv_block_0_hannpool'] # 0 is cue, 1 is mixture 
gains = gain_functions['attn0'](cue_acts)
attended_manual = mixture * gains
## Manual gain application should be the same as attended - it is close enough to numerical precision. 
# Can save gains this way for old models. Will make gains own op in future models and just with a normal hook.
print(torch.allclose(attended, attended_manual, atol=1e-128))

True

True

In [18]:
out_name =  Path(f'binaural_model_attn_stage_reps/{model_name}/{model_name}_layer_shape_dict.pkl')

shape_dict = {key:val.shape[1:] for key, val in activations.items()}
shape_dict['cochleagram'] = fg_cue.shape[1:]
print(shape_dict)
with open(out_name, 'wb') as f:
    pickle.dump(shape_dict, f)

{'conv_block_0': torch.Size([32, 40, 20000]), 'conv_block_1': torch.Size([64, 20, 5000]), 'conv_block_2': torch.Size([256, 10, 1250]), 'conv_block_3': torch.Size([512, 10, 250]), 'conv_block_4': torch.Size([512, 10, 63]), 'conv_block_5': torch.Size([512, 10, 63]), 'conv_block_6': torch.Size([512, 10, 63]), 'relufc': torch.Size([512]), 'cochleagram': torch.Size([2, 40, 20000])}


In [20]:
out_name

PosixPath('binaural_model_attn_stage_reps/word_task_half_co_loc_v07/word_task_half_co_loc_v07_layer_shape_dict.pkl')

In [ ]:
# h5_file = h5py.File(outname, 'r')
# print(h5_file.keys())
# # do corrs 
# layers = [key.split('_mixture')[0] for key in h5_file.keys() if 'mixture' in key]
# target_corrs = {}
# bg_corrs = {}
# for layer in layers:
#     fg_act = h5_file[f"{layer}_fg"]
#     bg_act = h5_file[f"{layer}_bg"]
#     mixture_act = h5_file[f"{layer}_mixture"]
#     n_acts = fg_act.shape[0]
#     target_corrs[layer] = [stats.pearsonr(fg_act[i], mixture_act[i])[0] for i in range(n_acts)]
#     bg_corrs[layer] = [stats.pearsonr(bg_act[i], mixture_act[i])[0] for i in range(n_acts)]

# h5_file.close()


: 

In [ ]:
import pandas as pd 

: 

In [ ]:
dfs = []
for layer in target_corrs.keys():
    df = pd.DataFrame.from_dict({'fg_corrs':target_corrs[layer],
                                 'bg_corrs':bg_corrs[layer],
                                 'layer': [layer] * len(target_corrs[layer])})
            
    dfs.append(df)
corr_results = pd.concat(dfs)

# melt fg_corrs and bg_corrs into one column
corr_results = corr_results.melt(id_vars=['layer'], value_vars=['fg_corrs', 'bg_corrs'], var_name='condition', value_name='correlation')

: 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


: 

: 

: 